# Actual Building Capacity Combinations Analysis

This notebook analyzes all **actually existing** combinations of geographical factors in the game world.

It uses the `building_analysis` package for data processing.

In [273]:
import os
import pandas as pd

from analysis.building_levels.building_analysis import CapacityAnalyzer, get_path

# Paths
data_dir = get_path("data_dir")
csv_path = os.path.join(data_dir, "actual_building_combinations.csv")

if not os.path.exists(csv_path):
    print(f"Error: {csv_path} not found. Please run scripts/analyze_actual.py first.")
else:
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} unique existing combinations.")
    
    building_cols = ["Fruit Orchard", "Sheep Farm", "Farming Village", "Fishing Village", "Forest Village"]
    df['Max Capacity'] = df[building_cols].max(axis=1)
    df['Total Capacity Sum'] = df[building_cols].sum(axis=1)
    
    display(df.head())

Loaded 446 unique existing combinations.


,Climate,Topography,Vegetation,Coastal,River,Lake,Farming Village,Fishing Village,Forest Village,Fruit Orchard,Sheep Farm,Location Count,Max Capacity,Total Capacity Sum
0,continental,flatland,woods,non_coastal,non_river,non_lake,13.0,4.0,12.0,7.0,6.0,951,13.0,42.0
1,tropical,flatland,jungle,non_coastal,non_river,non_lake,9.0,5.0,8.0,4.0,3.0,840,9.0,29.0
2,continental,flatland,grasslands,non_coastal,non_river,non_lake,16.0,4.0,8.0,8.0,11.0,784,16.0,47.0
3,continental,flatland,forest,non_coastal,non_river,non_lake,13.0,4.0,15.0,5.0,6.0,631,15.0,43.0
4,arctic,flatland,forest,non_coastal,non_river,non_lake,7.0,6.0,18.0,4.0,3.0,605,18.0,38.0


## Most Common Geographical Niches

In [274]:
df_common = df.sort_values('Location Count', ascending=False).head(40)
display(df_common[['Climate', 'Topography', 'Vegetation', 'Coastal', 'River', 'Lake', 'Location Count', 'Max Capacity', 'Total Capacity Sum'] + building_cols])

,Climate,Topography,Vegetation,Coastal,River,Lake,Location Count,Max Capacity,Total Capacity Sum,Fruit Orchard,Sheep Farm,Farming Village,Fishing Village,Forest Village
0,continental,flatland,woods,non_coastal,non_river,non_lake,951,13.0,42.0,7.0,6.0,13.0,4.0,12.0
1,tropical,flatland,jungle,non_coastal,non_river,non_lake,840,9.0,29.0,4.0,3.0,9.0,5.0,8.0
2,continental,flatland,grasslands,non_coastal,non_river,non_lake,784,16.0,47.0,8.0,11.0,16.0,4.0,8.0
3,continental,flatland,forest,non_coastal,non_river,non_lake,631,15.0,43.0,5.0,6.0,13.0,4.0,15.0
4,arctic,flatland,forest,non_coastal,non_river,non_lake,605,18.0,38.0,4.0,3.0,7.0,6.0,18.0
5,tropical,hills,jungle,non_coastal,non_river,non_lake,474,10.0,35.0,7.0,7.0,6.0,5.0,10.0
6,arid,flatland,sparse,non_coastal,non_river,non_lake,441,13.0,31.0,5.0,13.0,8.0,2.0,3.0
7,continental,hills,forest,non_coastal,non_river,non_lake,431,17.0,49.0,8.0,10.0,10.0,4.0,17.0
8,arid,flatland,desert,non_coastal,non_river,non_lake,409,11.0,28.0,5.0,11.0,7.0,2.0,3.0
9,continental,mountains,forest,non_coastal,non_river,non_lake,382,17.0,40.0,3.0,8.0,8.0,4.0,17.0


## Global Capacity Statistics

In [275]:
stats = []
for col in building_cols:
    weighted_sum = (df[col] * df['Location Count']).sum()
    total_locations = df['Location Count'].sum()
    weighted_avg = weighted_sum / total_locations
    stats.append({
        "Building": col,
        "Global Capacity": weighted_sum,
        "Weighted Avg Capacity": weighted_avg,
        "Max Possible": df[col].max(),
    })

display(pd.DataFrame(stats))
print(f"{int(round(pd.DataFrame(stats)['Global Capacity'].sum()*1000)):,}")

,Building,Global Capacity,Weighted Avg Capacity,Max Possible
0,Fruit Orchard,150291.0,6.572972,19.0
1,Sheep Farm,190556.0,8.333960,18.0
2,Farming Village,193759.0,8.474043,18.0
3,Fishing Village,100707.0,4.404417,14.0
4,Forest Village,200255.0,8.758146,20.0


835,568,000


## RGO (`raw_material`) by geography

Run the **load** code cell below once to load merged locations (slow). Then set **`RGO_SCOPE`** and **`RGO_GOODS`** in the **filter** code cell and run it — that step only filters and displays the cached data (fast).

`RGO_SCOPE` can be a **column** name (`super_region`, `macro_region`, `region`, …) or the string **`"global"`** for a **single row** showing the mix across all RGO locations. Rows are sorted by **`n_locations`** (desc), then by sum of displayed percentages. These scope labels are **dropped**: `atlantic_ocean_continent`, `south_atlantic_sub_continent`, `north_atlantic_ocean_sub_continent`.

Among locations with an RGO (`raw_material` other than `no_rgo`): percent share of each good within each scope bucket (from location templates). Locations without RGO are excluded. The first column **`n_locations`** is how many locations fall in that scope (or total for `global`). Good columns are ordered by **descending sum of column values** (sum of each good’s percentages across displayed rows). With **`RGO_GOODS`** as a list, only those goods are shown (still reordered by column sum). Values are shown as **percentages** with one decimal. Cells are **centered**; **green** applies to good columns only (row-wise); counts are not colored.

**Distinguishing goods** (following code cell): **`rgo_scope_distinguishing_df`** — one row per scope, columns **`rank_1` … `rank_N`** with good **names** (|Pearson residual| vs independence). **`rgo_good_top_scopes_df`** — inverted: one row per **good**, **`rank_*`** = scope names by **Pearson residual** (desc) for that good (e.g. rice → top region). **Colors:** good fills → **`rgo_good_fill_colors`**; scope fills (inverted) → **`rgo_scope_fill_colors`**. Requires `RGO_SCOPE` not `global`.


In [276]:
# Load merged locations (slow). Re-run when game/mod data changes.
analyzer_rgo = CapacityAnalyzer()
df_loc = analyzer_rgo.location_data.get_merged_df()
if df_loc.empty:
    _rgo_cached = None
    print("No location data loaded.")
else:
    d0 = df_loc.copy()
    d0["raw_material"] = d0["raw_material"].fillna("no_rgo")
    _rgo_cached = d0[d0["raw_material"] != "no_rgo"].copy()
    print(f"Loaded {len(_rgo_cached)} locations with RGO. Run the next cell to filter / display.")


Loading live game data...
Loaded 20929 locations with RGO. Run the next cell to filter / display.


In [302]:
# --- Asia --- # nice because pepper locks this to this region
khichdi = ["rice", "legumes", "pepper"]
fish_congee = ["rice", "fish", "pepper"] # Fixed: Swapped wild_game for fish

# --- Europe ---
mutton_pottage = ["wheat", "livestock", "saffron"] # saffron is more available
mediterranean_fish = ["fish", "olives", "wine"] # might lock this to non muslims because of wine?
mutton_and_pease = ["wool", "legumes", "saffron"] # saffron is more available
surstromming = ["fish", "millet", "salt"] # I dunno, salt is so available

# --- America ---
pozole = ["maize", "wild_game", "chili"]
pemmican = ["wild_game", "livestock", "fruit"] # Reordered to reflect wild_game dominance
labskaus = ["potato", "fish", "livestock"] # Note: relies on American potato RGOs!

# --- Africa / Middle East ---
saltfish_porridge = ["millet", "fish", "salt"]
meat_tajine = ["livestock", "fruit", "saffron"] # New: Uses Africa's high livestock and fruit

# --- Oceania ---
ika_mata = ["fish", "fruit", "salt"] # New: Uses Oceania's massive fish and fruit RGOs

farmed_staple_carbs = ["wheat", "rice", "maize", "millet", "potato", "legumes"]

animal_products = ["livestock", "wild_game", "fish", "wool", "fur", "beeswax"]

drinks = ["beer", "tea", "coffee", "cocoa", "liquor", "wine"]

spices = ["cloves", "pepper", "salt", "sugar", "chili", "saffron"]

all_base_goods = ["chili", "fish", "fruit", "legumes", "livestock", "maize", "millet", "olives", "pepper", "potato", "rice", "saffron", "salt", "wheat", "wine", "wild_game", "wool", "sugar", "beeswax", "fur"]

with_drinks = ["chili", "fish", "fruit", "legumes", "livestock", "maize", "millet", "olives", "pepper", "potato", "rice", "saffron", "salt", "wheat", "wine", "wild_game", "wool", "sugar", "beeswax", "fur", "tea", "beer", "coffee", "cocoa", "liquor"]

# Scope / goods (edit here, then run — uses data from the cell above; no re-parse)
RGO_SCOPE = "macro_region"  # "global" | super_region | macro_region | region | area | ...
# RGO_GOODS = None  # None or [] = all goods; else e.g. ["lumber", "fish"] (column order = list order)
# RGO_GOODS = ["saffron", "pepper", "chili", "cloves", "salt"] # spices
# RGO_GOODS = all_base_goods
# RGO_GOODS = ["cloves", "pepper", "chili", "saffron"]
RGO_GOODS = all_base_goods
RGO_DISTINGUISHING_N = 6

In [303]:

_rgo_base = globals().get("_rgo_cached")
if _rgo_base is None or len(_rgo_base) == 0:
    print("Run the cell above to load location data first.")
else:
    scope = RGO_SCOPE
    global_order = (
        _rgo_base["raw_material"]
        .value_counts()
        .sort_values(ascending=False)
        .index.tolist()
    )
    pct = None
    if scope == "global":
        d = _rgo_base
        if not d.empty:
            s = d["raw_material"].value_counts(normalize=True).mul(100)
            pct = pd.DataFrame(
                [s.reindex(global_order, fill_value=0).tolist()],
                index=["global"],
                columns=global_order,
            )
    else:
        if scope not in _rgo_base.columns:
            raise ValueError(
                f"RGO_SCOPE {scope!r} is not a column in merged locations. "
                f"Try 'global', super_region, macro_region, region, area, ..."
            )
        d = _rgo_base.dropna(subset=[scope])
        if not d.empty:
            pct = (
                d.groupby(scope)["raw_material"]
                .value_counts(normalize=True)
                .mul(100)
                .unstack(fill_value=0)
            )
            pct = pct.reindex(columns=global_order, fill_value=0)

    if pct is None or len(pct) == 0:
        if scope == "global":
            print("No RGO locations in cache.")
        else:
            print("No locations with RGO and a valid value for this RGO_SCOPE.")
    else:
        if scope == "global":
            n_series = pd.Series([len(d)], index=pct.index, name="n_locations")
        else:
            n_series = (
                d.groupby(scope).size().reindex(pct.index).fillna(0).astype(int)
            )
            n_series.name = "n_locations"
        pct = pd.concat([n_series, pct], axis=1)

        RGO_EXCLUDED_SCOPES = frozenset(
            {
                "atlantic_ocean_continent",
                "south_atlantic_sub_continent",
                "north_atlantic_ocean_sub_continent",
            }
        )
        pct = pct[~pct.index.isin(RGO_EXCLUDED_SCOPES)]
        if len(pct) == 0:
            print("No rows left after excluding ocean scopes.")
        else:
            goods_filter = RGO_GOODS
            if goods_filter is None or (
                isinstance(goods_filter, list) and len(goods_filter) == 0
            ):
                pct_display = pct
            else:
                known = set(global_order)
                col_order = list(dict.fromkeys(g for g in goods_filter if g in known))
                unknown = [g for g in goods_filter if g not in known]
                if unknown:
                    print(f"Warning: not in RGO data (ignored): {unknown}")
                if not col_order:
                    print("RGO_GOODS had no valid entries; showing all goods.")
                    pct_display = pct
                else:
                    pct_display = pct.loc[:, ["n_locations"] + col_order]

            good_cols = [c for c in pct_display.columns if c != "n_locations"]
            if good_cols:
                good_cols = (
                    pct_display[good_cols]
                    .sum(axis=0)
                    .sort_values(ascending=False)
                    .index.tolist()
                )
                pct_display = pct_display[["n_locations"] + good_cols]
            if good_cols:
                row_order = (
                    pct_display.assign(_pct_sum=pct_display[good_cols].sum(axis=1))
                    .sort_values(
                        by=["n_locations", "_pct_sum"],
                        ascending=[False, False],
                    )
                    .index
                )
            else:
                row_order = pct_display.sort_values(
                    by="n_locations", ascending=False
                ).index
            pct_display = pct_display.reindex(row_order)

            with pd.option_context(
                "display.max_columns", None,
                "display.width", None,
            ):
                sty = pct_display.style
                sty = sty.format("{:,.0f}", subset=["n_locations"])
                if good_cols:
                    sty = sty.format("{:.1f}%", subset=good_cols)
                sty = sty.set_properties(**{"text-align": "center"})
                sty = sty.set_table_styles(
                    [{"selector": "th", "props": [("text-align", "center")]}],
                    overwrite=False,
                )
                if good_cols:
                    sty = sty.background_gradient(
                        cmap="Greens", axis=1, subset=good_cols
                    )
                display(sty)


,n_locations,livestock,wild_game,fish,fruit,millet,legumes,wheat,fur,wool,rice,beeswax,salt,sugar,maize,olives,wine,saffron,pepper,chili,potato
macro_region,,,,,,,,,,,,,,,,,,,,,
east_asia,"3,146",16.1%,6.0%,6.6%,1.3%,3.8%,5.4%,6.9%,3.3%,2.3%,8.9%,2.1%,1.8%,0.5%,0.0%,0.0%,0.3%,0.0%,1.0%,0.0%,0.0%
north_america,"3,084",0.9%,20.2%,10.1%,2.8%,0.0%,4.2%,0.0%,11.1%,0.0%,0.0%,1.2%,1.2%,0.0%,6.5%,0.0%,0.0%,0.0%,0.0%,1.0%,0.0%
western_europe,"2,773",10.0%,4.9%,7.1%,3.0%,4.2%,3.4%,9.6%,2.4%,7.8%,0.1%,2.4%,3.0%,0.2%,0.0%,1.4%,4.5%,0.4%,0.0%,0.0%,0.0%
eastern_europe,"2,517",13.6%,5.6%,3.2%,1.9%,5.7%,4.1%,7.1%,6.6%,7.6%,0.0%,2.2%,2.2%,0.0%,0.0%,0.6%,2.1%,0.2%,0.0%,0.0%,0.0%
south_america,"1,357",3.2%,13.5%,9.3%,4.3%,0.0%,6.2%,0.0%,0.0%,2.1%,0.0%,4.0%,4.4%,0.0%,5.5%,0.0%,0.0%,0.0%,0.0%,1.9%,2.4%
middle_east,"1,262",13.2%,3.1%,4.4%,14.4%,1.0%,4.8%,9.8%,0.3%,7.3%,1.5%,1.8%,1.7%,1.9%,0.0%,1.2%,1.5%,1.2%,0.0%,0.0%,0.0%
south_asia,"1,230",7.9%,6.7%,2.4%,4.5%,5.4%,2.9%,3.7%,0.2%,4.2%,5.0%,1.5%,1.4%,2.8%,0.0%,0.0%,0.0%,0.0%,2.6%,0.0%,0.0%
south_east_asia,"1,044",4.1%,1.8%,15.9%,4.0%,5.6%,3.8%,0.0%,0.2%,0.4%,7.9%,2.8%,1.1%,0.9%,0.0%,0.0%,0.0%,0.0%,0.4%,0.0%,0.0%
north_asia,874,1.8%,32.2%,10.1%,0.0%,0.0%,0.1%,0.0%,25.3%,0.0%,0.0%,0.1%,1.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%


In [304]:
# RGO: top distinguishing goods per scope (vs independence)
# Requires: load cell, config (RGO_SCOPE, RGO_GOODS, all_base_goods), `_rgo_cached`.
# Set RGO_DISTINGUISHING_N to how many good-name columns (rank_1, rank_2, ...).
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import seaborn as sns


RGO_DISTINGUISHING_PALETTE = "tab20"  # high-contrast categorical; try tab10, Set1, bright, husl

RGO_EXCLUDED_SCOPES = frozenset(
    {
        "atlantic_ocean_continent",
        "south_atlantic_sub_continent",
        "north_atlantic_ocean_sub_continent",
    }
)

rgo_scope_distinguishing_df = pd.DataFrame()
rgo_good_top_scopes_df = pd.DataFrame()
rgo_good_fill_colors: dict[str, str] = {}  # good_id -> hex fill (updated when table runs)
rgo_scope_fill_colors: dict[str, str] = {}  # scope label -> hex fill (inverted table)


def _goods_columns_for_mix(ct: pd.DataFrame) -> list[str]:
    gf = globals().get("RGO_GOODS")
    if gf is None or (isinstance(gf, list) and len(gf) == 0):
        ab = globals().get("all_base_goods") or []
        if ab:
            return [g for g in ab if g in ct.columns]
        return sorted(ct.columns.tolist())
    return list(dict.fromkeys(g for g in gf if g in ct.columns))


def _color_map_for_goods(unique_goods: list[str], palette: str) -> dict[str, str]:
    """One color per good via sns.color_palette (same as hue= in seaborn plots)."""
    uniq = sorted(set(unique_goods))
    n = len(uniq)
    pal = sns.color_palette(palette, n_colors=max(n, 3))
    return {g: mcolors.to_hex(pal[i]) for i, g in enumerate(uniq)}


def _style_distinguishing_df(
    df: pd.DataFrame, fill: dict[str, str], *, label_col: str = "scope"
):
    # Styler.map requires unique index and columns (duplicate RGO_GOODS breaks this).
    df = df.copy()
    if not df.columns.is_unique:
        df = df.loc[:, ~df.columns.duplicated(keep="first")]
    if not df.index.is_unique:
        df = df.reset_index(drop=True)
    rank_cols = [c for c in df.columns if str(c).startswith("rank_")]

    def cell_style(x):
        if pd.isna(x) or x == "":
            return ""
        s = str(x)
        bg = fill.get(s, "#e8e8e8")
        return f"background-color: {bg}; text-align: center"

    sty = df.style.map(cell_style, subset=rank_cols)
    sty = sty.set_properties(**{"text-align": "left"}, subset=[label_col])
    return sty


_rgo_mix = globals().get("_rgo_cached")
if _rgo_mix is None or len(_rgo_mix) == 0:
    print("Run the load cell first to populate _rgo_cached.")
else:
    scope_col = RGO_SCOPE
    if scope_col == "global":
        print(
            "RGO_SCOPE is 'global'. Use a column like super_region "
            "(not 'global') to compare scopes."
        )
    elif scope_col not in _rgo_mix.columns:
        raise ValueError(
            f"RGO_SCOPE {scope_col!r} is not a column in merged locations."
        )
    else:
        dm = _rgo_mix.dropna(subset=[scope_col]).copy()
        dm = dm[~dm[scope_col].isin(RGO_EXCLUDED_SCOPES)]
        ct = pd.crosstab(dm[scope_col], dm["raw_material"])
        goods_cols = _goods_columns_for_mix(ct)
        if not goods_cols:
            print("No goods columns after filter; check RGO_GOODS.")
        else:
            ct = ct.reindex(columns=goods_cols, fill_value=0).sort_index()
            ct = ct.loc[:, ct.sum(axis=0) > 0]
            row_sums = ct.sum(axis=1)
            ct = ct[row_sums > 0]
            goods_cols = ct.columns.tolist()
            if len(goods_cols) == 0:
                print("No goods with positive counts after filtering.")
            elif len(ct) < 2:
                print("Need at least 2 scopes with RGO locations after exclusions.")
            else:
                O = ct.to_numpy(dtype=float)
                r = O.sum(axis=1)
                c = O.sum(axis=0)
                n = float(r.sum())
                E = np.outer(r, c) / n
                with np.errstate(divide="ignore", invalid="ignore"):
                    pr = (O - E) / np.sqrt(E)
                pr = np.nan_to_num(pr, nan=0.0, posinf=0.0, neginf=0.0)
                scope_labels = ct.index.tolist()
                good_names = ct.columns.tolist()
                wide_rows = []
                N = int(RGO_DISTINGUISHING_N)
                for i, sc in enumerate(scope_labels):
                    order = np.argsort(-np.abs(pr[i, :]))
                    row = {"scope": sc}
                    for k, j in enumerate(order[:N], start=1):
                        row[f"rank_{k}"] = good_names[j]
                    wide_rows.append(row)
                rgo_scope_distinguishing_df = pd.DataFrame(wide_rows)
                rank_cols = [c for c in rgo_scope_distinguishing_df.columns if str(c).startswith("rank_")]
                uniq_vals = [
                    str(v)
                    for col in rank_cols
                    for v in rgo_scope_distinguishing_df[col].dropna().unique()
                ]
                fill = _color_map_for_goods(uniq_vals, RGO_DISTINGUISHING_PALETTE)
                globals()["rgo_good_fill_colors"] = fill
                print(
                    f"Columns rank_1 ... rank_{N}: good names in order of |Pearson residual| "
                    "(independence model). Same good = same fill color."
                )
                display(_style_distinguishing_df(rgo_scope_distinguishing_df, fill))
                inv_rows = []
                n_scopes = len(scope_labels)
                K = min(N, n_scopes)
                for j, gname in enumerate(good_names):
                    order = np.argsort(-pr[:, j])
                    row = {"good": gname}
                    for k in range(1, K + 1):
                        row[f"rank_{k}"] = scope_labels[order[k - 1]]
                    inv_rows.append(row)
                rgo_good_top_scopes_df = pd.DataFrame(inv_rows)
                inv_rank_cols = [
                    c for c in rgo_good_top_scopes_df.columns if str(c).startswith("rank_")
                ]
                uniq_scopes = sorted(
                    {
                        str(v)
                        for col in inv_rank_cols
                        for v in rgo_good_top_scopes_df[col].dropna().unique()
                    }
                )
                fill_s = _color_map_for_goods(uniq_scopes, RGO_DISTINGUISHING_PALETTE)
                globals()["rgo_scope_fill_colors"] = fill_s
                print(
                    f"Inverted: rows=goods, rank_1 ... rank_{K}: scopes by Pearson residual "
                    "(desc) for that good. Same scope = same fill color."
                )
                display(
                    _style_distinguishing_df(
                        rgo_good_top_scopes_df, fill_s, label_col="good"
                    )
                )


Columns rank_1 ... rank_6: good names in order of |Pearson residual| (independence model). Same good = same fill color.


,scope,rank_1,rank_2,rank_3,rank_4,rank_5,rank_6
0,australasia,millet,wild_game,wheat,fur,wool,rice
1,central_africa,wild_game,livestock,fur,wool,rice,fish
2,central_asia,livestock,wheat,fish,rice,fur,millet
3,east_africa,millet,livestock,fur,saffron,wool,wild_game
4,east_asia,rice,livestock,fruit,wild_game,maize,pepper
5,eastern_europe,wool,fish,rice,wild_game,wheat,maize
6,middle_east,fruit,wild_game,wheat,fur,wool,saffron
7,north_africa,olives,sugar,fruit,wild_game,fur,wool
8,north_america,maize,wild_game,fur,livestock,wheat,millet
9,north_asia,fur,wild_game,livestock,wheat,fruit,legumes


Inverted: rows=goods, rank_1 ... rank_6: scopes by Pearson residual (desc) for that good. Same scope = same fill color.


,good,rank_1,rank_2,rank_3,rank_4,rank_5,rank_6
0,chili,south_america,north_america,southern_africa,pacific_islands,north_africa,central_africa
1,fish,south_east_asia,pacific_islands,north_america,australasia,south_america,east_africa
2,fruit,middle_east,pacific_islands,north_africa,south_asia,east_africa,south_east_asia
3,legumes,south_america,east_asia,australasia,pacific_islands,south_east_asia,central_asia
4,livestock,east_asia,central_asia,west_africa,eastern_europe,central_africa,east_africa
5,maize,north_america,south_america,southern_africa,pacific_islands,north_africa,central_africa
6,millet,australasia,west_africa,east_africa,eastern_europe,south_east_asia,south_asia
7,olives,north_africa,western_europe,middle_east,eastern_europe,southern_africa,pacific_islands
8,pepper,south_asia,east_asia,south_east_asia,southern_africa,pacific_islands,north_africa
9,potato,south_america,southern_africa,pacific_islands,north_africa,central_africa,central_asia


In [ ]:
# farmed_staple_carbs = ["wheat", "rice", "maize", "millet", "potato", "legumes"]
# europe: wheat
# asia: rice
# america: maize / potato
# africa: millet
# oceania:  maybe millet

animal_products = ["livestock", "wild_game", "fish", "wool", "fur", "beeswax"]
# europe: 